**How would we translate a query like this?**
``` sql
SELECT * 
FROM stuff
WHERE a=7
```

**Create some fake data**


In [1]:
from faker import Faker

fake = Faker()
fake.seed_instance(42)

from dataclasses import dataclass

number_of_tuples = 40


@dataclass
class Stuff:
    a: int
    b: int


# create some fake data mimicking tuples in a relation:
data = [
    Stuff(fake.pyint(max_value=10), fake.pyint(max_value=20))
    for f in range(number_of_tuples)
]
data[:10]

[Stuff(a=10, b=3),
 Stuff(a=0, b=8),
 Stuff(a=3, b=7),
 Stuff(a=2, b=3),
 Stuff(a=10, b=17),
 Stuff(a=1, b=18),
 Stuff(a=6, b=1),
 Stuff(a=0, b=2),
 Stuff(a=3, b=7),
 Stuff(a=8, b=19)]

**Algorithm 1: Scan per Query**

In [2]:
def scan(data, attribute, constant):
    res = []
    for el in data:
        if el.__dict__[attribute] == constant:
            res.append(el)
    return res


scan(data, "a", 7)

[Stuff(a=7, b=20)]


**What if we have more queries?**
``` sql
SELECT * 
FROM stuff
WHERE a=7
```
``` sql
SELECT * 
FROM stuff
WHERE b=12
```
``` sql
SELECT * 
FROM stuff
WHERE b=8
```


**Algorithm 2: One scan()-call per query**

In [3]:
# the queries we want to run:
queries_dict = {
    "q1": {"attribute": "a", "constant": 7},
    "q2": {"attribute": "b", "constant": 12},
    "q3": {"attribute": "b", "constant": 8},
}

for key, value in queries_dict.items():
    print(key, scan(data, value["attribute"], value["constant"]))

q1 [Stuff(a=7, b=20)]
q2 []
q3 [Stuff(a=0, b=8), Stuff(a=9, b=8), Stuff(a=9, b=8), Stuff(a=6, b=8), Stuff(a=3, b=8)]


**Algorithm 3: Shared Scan**

In [4]:
def shared_scan(data, queries_dict):
    # create a result dictionary with one result list per incoming query:
    res = {k: [] for k in queries_dict}

    # loop over all tuples:
    for el in data:
        # check for each query...
        for query, entry in queries_dict.items():
            # whether this tuple is part of the result:
            if el.__dict__[entry["attribute"]] == entry["constant"]:
                # append to dictionary:
                res[query].append(el)
    return res


shared_scan(data, queries_dict)

{'q1': [Stuff(a=7, b=20)],
 'q2': [],
 'q3': [Stuff(a=0, b=8),
  Stuff(a=9, b=8),
  Stuff(a=9, b=8),
  Stuff(a=6, b=8),
  Stuff(a=3, b=8)]}

**Unit Tests**

In [5]:
import unittest


class MyTest(unittest.TestCase):

    # the following function will be called before every test-function:

    def create_data(self, number_of_tuples):
        self.data = [
            Stuff(fake.pyint(max_value=20), fake.pyint(max_value=20))
            for f in range(number_of_tuples)
        ]

    def test_scans_end_to_end(self):
        for i in range(10):
            self.create_data(1000)

            # a dictionary with queries
            _queries_dict = {
                "q"
                + str(i): {
                    "attribute": "a" if fake.pybool() else "b",
                    "constant": fake.pyint(max_value=20),
                }
                for i in range(30)
            }
            # compute the result for all these queries by calling the scan()-method for each query inbdependently:
            res1_dict = {
                query: scan(self.data, entry["attribute"], entry["constant"])
                for query, entry in _queries_dict.items()
            }

            # single call to the shared_scan-method:
            res2_dict = shared_scan(self.data, _queries_dict)

            self.assertEqual(res1_dict, res2_dict)


# Run the unit test without shutting down the jupyter kernel
# notice that this call will look for all test classes, i.e.
# all classes inheriting from unittest.TestCase
# each of these functions will be executed once
# unittest.main(argv=['ignored', '-v'], verbosity=2, exit=False)

# only execute a specific Test-class:
unittest.main(argv=["ignored", "-v"], defaultTest="MyTest", verbosity=2, exit=False)

test_scans_end_to_end (__main__.MyTest.test_scans_end_to_end) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.061s

OK
